# PTB-XL ECG Intelligence: Staged Research Pipeline

Comprehensive end-to-end multi-label ECG intelligence pipeline on PTB-XL (100 Hz).
Each stage is independently executable, logged, and structured to prevent data leakage.

--- 
## Stage 0: Environment & Setup
- Mount Google Drive
- Unpack project and enter directory
- Install editable package with dependencies
- Verify Python, PyTorch, CUDA, and package imports

In [ ]:
# Stage 0.1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Stage 0.2: Extract project package and enter root
# Expected archive: MyDrive/ptbxl-ecg-intelligence-colab.zip
!unzip -q -o /content/drive/MyDrive/ptbxl-ecg-intelligence-colab.zip -d /content/ptbxl-ecg-intelligence

%cd /content/ptbxl-ecg-intelligence
!pwd && ls -lh

In [ ]:
# Stage 0.3: Install project in editable mode and verify environment
!pip -q install wfdb PyWavelets xgboost scikit-learn
!pip -q install --no-deps -e .

import sys, torch, ecg_intelligence
print("=" * 60)
print(f"Python Version   : {sys.version.split()[0]}")
print(f"PyTorch Version  : {torch.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device       : {torch.cuda.get_device_name(0)}")
print(f"Package Version  : {ecg_intelligence.__version__}")
print("=" * 60)

--- 
## Stage 1: Dataset Audit & Fast Disk Copy
- Check Google Drive PTB-XL location
- Copy records100 & CSVs to fast local SSD (/content/data/ptbxl) to avoid Drive FUSE latency
- Run dataset integrity audit (metadata, multi-label prevalence, patient fold isolation, sample WFDB waveforms)

In [ ]:
# Stage 1.1: Copy dataset from Drive to fast local VM disk
import os, time

DRIVE_PTBXL = '/content/drive/MyDrive/ptb-xl/1.0.3'
LOCAL_PTBXL = '/content/data/ptbxl'
DRIVE_CACHE = '/content/drive/MyDrive/ptbxl_ecg_intelligence/cache'  # Survives VM disconnection

os.environ['PTBXL_ROOT'] = LOCAL_PTBXL
os.environ['PTBXL_CACHE'] = DRIVE_CACHE

t0 = time.time()
!mkdir -p $LOCAL_PTBXL
print('Copying metadata CSVs...')
!cp $DRIVE_PTBXL/*.csv $LOCAL_PTBXL/

print('Copying records100 to local disk...')
!cp -r $DRIVE_PTBXL/records100 $LOCAL_PTBXL/
print(f'Done copying to {LOCAL_PTBXL} in {time.time() - t0:.1f}s')

In [ ]:
# Stage 1.2: Audit dataset integrity, label prevalences, patient fold leakage, and sample signals
!python scripts/audit_dataset.py --root $LOCAL_PTBXL --samples 5

--- 
## Stage 2 & 3: Signal Preprocessing & Handcrafted Feature Extraction
- Preprocess 100 Hz ECG waveforms
- Extract 139 handcrafted features (wavelet band energies, lead statistics, HRV)
- Persist cache (X.npy, F.npy, Y.npy, fold.npy) to Google Drive for fast subsequent runs

In [ ]:
# Stage 2 & 3: Build or verify cache
import time
from ecg_intelligence.data.cache import load_or_build_cache

t0 = time.time()
print(f"Checking/building preprocessed cache at: {DRIVE_CACHE}")
X, F, Y, fold = load_or_build_cache(root=LOCAL_PTBXL, cache_dir=DRIVE_CACHE, n_jobs=16)
print(f"Cache ready in {time.time() - t0:.1f}s:")
print(f"  X (raw signals)    : {X.shape} (dtype: {X.dtype})")
print(f"  F (features)       : {F.shape} (dtype: {F.dtype})")
print(f"  Y (targets)        : {Y.shape} (dtype: {Y.dtype})")
print(f"  fold (assignments) : {fold.shape} (folds: {sorted(list(set(fold)))})")

--- 
## Stage 4: Baseline Experiments (XGBoost & 1D ResNet)
- Train Multi-Label XGBoost on handcrafted features (F)
- Train 1D Deep ResNet on raw signals (X)
- Evaluate on official test fold 10 with thresholds tuned on fold 9

In [ ]:
# Stage 4.1: Train XGBoost baseline
!python experiments/train_xgb.py --root $LOCAL_PTBXL --cache $DRIVE_CACHE

In [ ]:
# Stage 4.2: Train 1D ResNet baseline
!python experiments/train_resnet.py --root $LOCAL_PTBXL --cache $DRIVE_CACHE --epochs 30 --bs 128

--- 
## Stage 5: Sequential Model (ResNet + BiGRU + Temporal Attention)
- Incorporates bidirectional temporal dynamics and attention pooling over cardiac cycles

In [ ]:
# Stage 5: Train Sequential Architecture
!python experiments/train_sequence.py --root $LOCAL_PTBXL --cache $DRIVE_CACHE --epochs 30 --bs 128

--- 
## Stage 6: Hybrid Models (Feature Concatenation vs. Adaptive Gated Fusion)
- Hybrid Concatenation: Standard linear concatenation of deep + handcrafted vectors
- Adaptive Gated Fusion: Proposed dynamically learned sigmoid gating mechanism

In [ ]:
# Stage 6.1: Train Feature Concatenation Hybrid model
!python experiments/train_hybrid.py --root $LOCAL_PTBXL --cache $DRIVE_CACHE --epochs 30 --bs 128

In [ ]:
# Stage 6.2: Train Adaptive Gated Fusion model (Proposed Method)
!python experiments/train_gated.py --root $LOCAL_PTBXL --cache $DRIVE_CACHE --epochs 30 --bs 128

--- 
## Stage 7: Probability-level Ensemble
- Blend predictions of Adaptive Gated Fusion and XGBoost strictly using validation weights

In [ ]:
# Stage 7: Evaluate Ensemble (Gated + XGBoost)
!python experiments/train_ensemble.py --cache $DRIVE_CACHE --ens_with gated

--- 
## Stage 8: Final Evaluation & Ablation Comparison Table
- Print the complete ablation comparison across all models
- Output Macro AUROC, Macro AUPRC, Macro F1, Macro Sensitivity, and Macro Specificity
- Export comparison.csv

In [ ]:
# Stage 8: Generate comparison table and verify comparison.csv
from ecg_intelligence.evaluation.ablation import format_ablation_markdown_table
import pandas as pd

csv_out = f"{DRIVE_CACHE}/comparison.csv"
tbl = format_ablation_markdown_table(DRIVE_CACHE, save_csv_path=csv_out)
print(tbl)

if os.path.exists(csv_out):
    print(f"\nSaved metrics to {csv_out}:")
    df_comp = pd.read_csv(csv_out)
    display(df_comp)

--- 
## Stage 9: Explainability & Faithfulness Verification
- Compute Integrated Gradients on test cases for key diagnostic superclasses (e.g., MI, STTC)
- Run deletion-vs-random faithfulness test to verify attributions truly reflect model decisions
- Plot lead attribution heatmaps

In [ ]:
# Stage 9.1: Run Deletion Faithfulness test on trained Gated model
import torch, json
import numpy as np
from ecg_intelligence.models.classifier import build_torch_model
from ecg_intelligence.explainability.deletion_test import run_deletion_faithfulness_test
from ecg_intelligence.preprocessing.normalization import normalize_signals, normalize_features

norm_data = np.load(f"{DRIVE_CACHE}/norm.npz")
Xn = normalize_signals(X, norm_data['mu'], norm_data['sd'])
Fn = normalize_features(F, norm_data['fm'], norm_data['fs'])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_path = f"{DRIVE_CACHE}/model_gated.pt"

if os.path.exists(model_path):
    net = build_torch_model(num_features=F.shape[1], mode='gated').to(device)
    net.load_state_dict(torch.load(model_path, map_location=device))
    net.eval()

    te_mask = (fold == 10)
    faith_res = run_deletion_faithfulness_test(
        net=net, Xn=Xn, Fn=Fn, Y=Y, test_mask=te_mask,
        target_class_str='MI', num_samples=30, top_fraction=0.10, device=device
    )
    print("Deletion Faithfulness Results:")
    print(json.dumps(faith_res, indent=2))
else:
    print(f"Model checkpoint {model_path} not found. Please run Stage 6.2 first.")

In [ ]:
# Stage 9.2: Render waveform attribution heatmap for a test case
import matplotlib.pyplot as plt
from ecg_intelligence.explainability.integrated_gradients import compute_integrated_gradients
from ecg_intelligence.explainability.visualization import plot_attribution_lead

if os.path.exists(model_path):
    te_indices = np.where(fold == 10)[0]
    idx = te_indices[0]
    x_sample = torch.from_numpy(Xn[idx:idx+1]).to(device)
    f_sample = torch.from_numpy(Fn[idx:idx+1]).to(device)

    ig = compute_integrated_gradients(net, x_sample, f_sample, target_class=2, steps=40)  # Class 2: MI
    ig_lead_v2 = ig[7].cpu().numpy()  # V2 lead (index 7)

    fig = plot_attribution_lead(X[idx], ig_lead_v2, lead_idx=7, title="Lead V2 - Integrated Gradients Attribution for Myocardial Infarction (MI)")
    plt.show()
else:
    print("Model checkpoint not found. Skipping plot.")

--- 
## Stage 10: Inference & UI Preparation
- Validate that model weights, norm statistics, and pre-extracted samples exist in the cache
- Test local API predictions without launching a full browser server

In [ ]:
# Stage 10: Test inference pipeline
from ecg_intelligence.inference.pipeline import ECGPredictor

print(f"Initializing ECGPredictor from cache: {DRIVE_CACHE}")
predictor = ECGPredictor(cache_dir=DRIVE_CACHE, mode='gated')

if predictor.is_loaded:
    sample_ecg = X[np.where(fold == 10)[0][0]]  # (1000, 12)
    res = predictor.predict(sample_ecg)
    print("\nInference successful!")
    print("Predicted Probabilities:", res['probabilities'])
    print("Binary Decisions       :", res['predictions'])
    if 'gate_weight' in res:
        print("Deep Branch Gate Weight:", res['gate_weight'])
else:
    print("\nPredictor in demo mode (artifacts not yet trained). UI will display notice banner.")